In [ ]:
class mac_top:
    def __init__(self, size):
        self.size     = size
        self.weights  = None   # loaded from B_tile each call
        self.psum_acc = None   # internal accumulator (T×T)

    def reset_accumulator(self):
        T = self.size
        self.psum_acc = [[0] * T for _ in range(T)]

    def run(self, A_tile, B_tile, A_valid, B_valid, store=0 , bias_valid = 0 ,  bias = None):
        """
        Unified MAC top module.
        Inputs:
            A_tile  : T×T activation matrix
            B_tile  : T×T weight matrix
            A_valid : 1 if A_tile is valid
            B_valid : 1 if B_tile is valid

            store   : 1 = accumulate internally, suppress output
                      0 = finalize, add bias, release output
            bias    : T×T bias matrix (added at finalization when store=0)
            bias_valid :  this  is valid signal for bias 
        Outputs:
            psum         : T×T result matrix (zeros if store=1)
            psum_valid   : 1 if psum is valid this cycle
            ready        : 1 if module is ready (not computing)
        """
        T = self.size

        psum       = [[0] * T for _ in range(T)]
        psum_valid = 0
        ready      = 0   # busy during compute

        # Both inputs must be valid
        if A_valid and B_valid:

            # Load weights from B_tile this cycle
            self.weights = [row[:] for row in B_tile]

            # Initialize accumulator if this is the first tile in a group
            if self.psum_acc is None:
                self.reset_accumulator()

            # MAC: psum_acc += A_tile × B_tile
            for i in range(T):
                for j in range(T):
                    for k in range(T):
                        self.psum_acc[i][j] += A_tile[i][k] * self.weights[k][j]

            if store == 1:
                # Accumulate internally — hold output
                psum       = [[0] * T for _ in range(T)]
                psum_valid = 0

            else:  # store == 0 → finalize
                # Add bias if provided
                if bias is not None and bias_valid:
                    for i in range(T):
                        for j in range(T):
                            self.psum_acc[i][j] += bias[i][j]

                # Release output and clear accumulator
                psum       = [row[:] for row in self.psum_acc]
                psum_valid = 1
                self.psum_acc = None   # reset for next tile group

        # Done — back to ready
        ready = 1
        return psum, psum_valid, ready
    
mac = mac_top(4)
A = [
     [1 , 2, 3 , 4     ],
     [  5, 6, 7,    8   ],
     [ 9 , 10 , 11 , 12 ],
     [ 13 , 14 , 15 , 16 ]
 ]

B = [
     [1 , 2, 3 , 4     ],
     [  5, 6, 7,    8   ],
     [ 9 , 10 , 11 , 12 ],
     [ 13 , 14 , 15 , 16 ]
 ]

mac.run( A_tile= A, B_tile= B, A_valid = 1, B_valid = 1 , store=0 , bias_valid = 0 ,  bias = None)



([[90, 100, 110, 120],
  [202, 228, 254, 280],
  [314, 356, 398, 440],
  [426, 484, 542, 600]],
 1,
 1)